# Claude Agent SDK: persisting sessions to external storage

By default the SDK writes session transcripts to JSONL files under `~/.claude/projects/` **on the local disk**. That is perfect on your laptop and a problem anywhere else: serverless functions, autoscaled workers and CI runners **don't share a filesystem**, and containers are ephemeral.

A **`SessionStore`** adapter solves this. It **mirrors** every transcript to a backend you choose (object store, Redis, Postgres...), so a session created on one host can be **resumed on another**.

**What you will learn (and see running):**
1. The `SessionStore` interface: two required methods, four optional ones
2. Build your own store and **test it with no API calls**
3. Why `append` must **deduplicate** (retries re-deliver entries)
4. Run the official **conformance suite** against your adapter
5. Plug a store into a real agent and watch the mirror fill up
6. **Resume from the store** as if on a different machine
7. Behavior notes: dual-write, best-effort mirroring, retention

> Read `agent-sdk-sessions-continue-resume-fork.ipynb` first if "session", "resume" and "fork" are new words.

## The idea in one picture

```
   host A                                           your backend
 ┌──────────────┐   1. subprocess writes      ┌───────────────────┐
 │ agent run    │ ──── local JSONL first ───▶ │                   │
 │              │   2. SDK forwards the same  │  SessionStore     │
 │              │ ──── batch to append() ───▶ │  (S3, Redis, PG…) │
 └──────────────┘                              └─────────┬─────────┘
                                                         │ load()
   host B  (different machine, same working directory)   │
 ┌──────────────┐   resume=<session_id>                  │
 │ agent run    │ ◀──────────────────────────────────────┘
 └──────────────┘   SDK loads transcript BEFORE the subprocess starts
```

The store is a **mirror** of the local transcript, not a replacement for it.

## 0. Setup

Run these four cells first. They check Python, install the SDK, load your API key from a `.env` file, and create a private working folder for the demos.

In [ ]:
import sys
print(sys.version)
assert sys.version_info >= (3, 10), "The Agent SDK needs Python 3.10 or newer"

In [ ]:
%pip install -q claude-agent-sdk python-dotenv pytest

Create a file named `.env` next to this notebook containing one line (no quotes, no spaces around `=`):

```
ANTHROPIC_API_KEY=your-api-key
```

In [ ]:
import os
from pathlib import Path
from dotenv import load_dotenv, find_dotenv

env_path = find_dotenv(usecwd=True)
load_dotenv(env_path, override=True)

key = os.environ.get("ANTHROPIC_API_KEY", "")
if not key:
    raise RuntimeError(
        f"ANTHROPIC_API_KEY not found. Create a .env file in {Path.cwd()} with:\n"
        "ANTHROPIC_API_KEY=your-api-key"
    )
print("Loaded .env from:", env_path)
print("API key loaded:", key[:7] + "..." + key[-4:])

In [ ]:
from pathlib import Path

DEMO_DIR = Path("session-storage-demo").resolve()
DEMO_DIR.mkdir(exist_ok=True)
print("Demo folder:", DEMO_DIR)

## 1. The `SessionStore` interface

A store is an object with **two required** and **four optional** async methods.

| Method | Required | Called when |
|---|:-:|---|
| `append(key, entries)` | yes | After each batch of transcript entries is written locally |
| `load(key)` | yes | Before the subprocess starts, when you `resume` (or `continue`). Return `None` if the session is unknown |
| `list_sessions(project_key)` | no | By `list_sessions_from_store` and by `continue_conversation=True`. Without it, continue **raises** |
| `list_session_summaries(project_key)` | no | To read metadata for all sessions in one call. Without it, listing falls back to `list_sessions` plus a `load` per session |
| `delete(key)` | no | By `delete_session_via_store`. Deleting the main key must **cascade** to all subkeys. Undefined = no-op (fine for append-only backends) |
| `list_subkeys(key)` | no | During resume, to find **subagent** transcripts. Without it only the main transcript is restored |

### What is a `SessionKey`?

```python
{"project_key": "...", "session_id": "...", "subpath": "..."}   # subpath is optional
```

- `project_key`: a filesystem-safe encoding of the **working directory**. That is why you must resume from a `cwd` that matches the original run.
- `session_id`: the session UUID.
- `subpath`: set only for **subagent transcripts or sidecar files**, e.g. `"subagents/agent-<id>"`. Treat it as an opaque suffix. **No `subpath` means the main transcript.**

### Rules for entries
Entries are JSON-safe dicts (`{"type": ..., ...}`). Treat them as **opaque**: persist them **in order** and return them in the **same order**. `load` must return values *deep-equal* to what was appended (byte-equal serialization is not required, so a backend that reorders keys is fine).

## 2. Build a store: JSONL files on disk

The simplest real backend is the filesystem itself. Our adapter stores one folder per session:

```
store/<project_key>/<session_id>/__main__.jsonl                 ← main transcript
store/<project_key>/<session_id>/subagents/agent-1.jsonl         ← a subagent subpath
```

It implements the two required methods plus `list_sessions`, `delete` and `list_subkeys`, and it **deduplicates by `uuid`** inside `append` (explained in section 4).

> A production adapter must also **serialize concurrent `append` calls for the same session** (a lock or transaction). This demo is single-writer.

In [ ]:
# OFFLINE
import json, shutil
from pathlib import Path
from claude_agent_sdk import SessionStore


class JsonlFileSessionStore(SessionStore):
    """Mirror session transcripts into JSONL files under `root`."""

    MAIN = "__main__.jsonl"

    def __init__(self, root, dedupe=True):
        self.root = Path(root)
        self.dedupe = dedupe
        self.root.mkdir(parents=True, exist_ok=True)

    # ---- key -> file -------------------------------------------------
    def _file(self, key):
        folder = self.root / key["project_key"] / key["session_id"]
        sub = key.get("subpath")
        return folder / self.MAIN if not sub else folder / f"{sub}.jsonl"

    # ---- required ----------------------------------------------------
    async def append(self, key, entries):
        path = self._file(key)
        path.parent.mkdir(parents=True, exist_ok=True)

        seen = set()
        if self.dedupe and path.exists():
            for line in path.open():
                uuid = json.loads(line).get("uuid")
                if uuid is not None:
                    seen.add(uuid)

        with path.open("a") as f:
            for entry in entries:
                uuid = entry.get("uuid")
                if self.dedupe and uuid is not None:
                    if uuid in seen:
                        continue          # already stored: a retried batch re-delivered it
                    seen.add(uuid)
                f.write(json.dumps(entry) + "\n")

    async def load(self, key):
        path = self._file(key)
        if not path.exists():
            return None                   # unknown session
        return [json.loads(line) for line in path.open()]

    # ---- optional ----------------------------------------------------
    async def list_sessions(self, project_key):
        found = []
        for folder in (self.root / project_key).glob("*"):
            main = folder / self.MAIN
            if main.exists():
                found.append({"session_id": folder.name,
                              "mtime": int(main.stat().st_mtime * 1000)})   # epoch ms
        return found

    async def delete(self, key):
        path = self._file(key)
        if key.get("subpath"):
            path.unlink(missing_ok=True)                    # only that subkey
        else:
            shutil.rmtree(path.parent, ignore_errors=True)  # cascade to every subkey

    async def list_subkeys(self, key):
        folder = self.root / key["project_key"] / key["session_id"]
        if not folder.exists():
            return []
        return sorted(
            p.relative_to(folder).as_posix()[: -len(".jsonl")]
            for p in folder.rglob("*.jsonl") if p.name != self.MAIN
        )


print("JsonlFileSessionStore defined")

## 3. Test it *without any API calls*

Because a store is just an object with async methods, you can exercise it directly. These checks encode the contract: order preserved, unknown sessions give `None`, subpaths are independent, delete cascades.

In [ ]:
# OFFLINE
import tempfile

tmp = Path(tempfile.mkdtemp(prefix="store-test-"))
store = JsonlFileSessionStore(tmp)

KEY = {"project_key": "proj", "session_id": "sess-1"}
SUB = {**KEY, "subpath": "subagents/agent-1"}

# 1. Order is preserved across batches
await store.append(KEY, [{"type": "user", "uuid": "b", "n": 1}, {"type": "assistant", "uuid": "a", "n": 2}])
await store.append(KEY, [{"type": "user", "uuid": "c", "n": 3}])
loaded = await store.load(KEY)
print("order kept        :", [e["uuid"] for e in loaded] == ["b", "a", "c"])

# 2. Unknown session -> None
print("unknown -> None   :", await store.load({"project_key": "proj", "session_id": "nope"}) is None)

# 3. Subpath is stored independently of the main transcript
await store.append(SUB, [{"type": "assistant", "uuid": "s1"}])
print("main untouched    :", len(await store.load(KEY)) == 3)
print("subpath separate  :", [e["uuid"] for e in await store.load(SUB)] == ["s1"])
print("list_subkeys      :", await store.list_subkeys(KEY))

# 4. list_sessions only reports main transcripts
print("list_sessions     :", [s["session_id"] for s in await store.list_sessions("proj")])

# 5. Deleting the main key cascades to the subkey
await store.delete(KEY)
print("main deleted      :", await store.load(KEY) is None)
print("subkey cascaded   :", await store.load(SUB) is None)

shutil.rmtree(tmp)

## 4. Why `append` must deduplicate

If `append()` **raises**, the SDK retries the batch **up to two more times** (three attempts in total) with a short backoff. The danger: an attempt can *succeed in writing* and *still raise* (a network timeout after the data landed, say). The retry then **re-delivers entries that are already stored**.

The docs' advice: **deduplicate by `entry.uuid` inside `append()`**.

Let's reproduce the failure on purpose: a batch is appended, an exception is raised afterwards, and the "SDK" retries.

In [ ]:
# OFFLINE
async def deliver_with_retry(store, key, batch, attempts=3):
    """Mimic the SDK: retry a batch when append() raises."""
    for attempt in range(1, attempts + 1):
        try:
            await store.append(key, batch)
            return attempt
        except ConnectionError:
            continue
    return None


def make_flaky(store_cls, **kwargs):
    """A store whose first append LANDS the data and then raises (timeout after write)."""
    class Flaky(store_cls):
        calls = 0
        async def append(self, key, entries):
            await super().append(key, entries)
            Flaky.calls += 1
            if Flaky.calls == 1:
                raise ConnectionError("timeout after the write succeeded")
    return Flaky(**kwargs)


BATCH = [{"type": "user", "uuid": "u1"}, {"type": "assistant", "uuid": "u2"}]

for dedupe in (False, True):
    tmp = Path(tempfile.mkdtemp(prefix="dedupe-"))
    s = make_flaky(JsonlFileSessionStore, root=tmp, dedupe=dedupe)
    await deliver_with_retry(s, KEY, BATCH)
    stored = [e["uuid"] for e in await s.load(KEY)]
    print(f"dedupe={dedupe!s:5} -> stored {stored}")
    shutil.rmtree(tmp)

Without dedupe the transcript contains **each entry twice**: a corrupted history on resume. With dedupe it is stored once.

> There is one more consequence of best-effort delivery: if all three attempts fail, the SDK **logs it, emits a `{"type": "system", "subtype": "mirror_error"}` message into your stream, drops the batch and keeps going**. The agent is never interrupted by a store outage (it writes locally first), so **monitor for `mirror_error`** if losing mirrored data matters.

## 5. Run the official conformance suite

The SDK ships a suite that asserts the full behavioral contract of `append`, `load` and each optional method (tests for optional methods **skip automatically** when you don't implement them). You pass a **zero-argument factory** that returns a **fresh, empty** store every time.

In [ ]:
# OFFLINE
from claude_agent_sdk.testing import run_session_store_conformance

def make_store():
    return JsonlFileSessionStore(tempfile.mkdtemp(prefix="conformance-"))

await run_session_store_conformance(make_store)
print("Conformance suite passed")

In a real project you would run this from pytest:

```python
import pytest
from claude_agent_sdk.testing import run_session_store_conformance

@pytest.mark.anyio
async def test_my_store_conformance():
    await run_session_store_conformance(lambda: MyRedisStore(client=fake_client()))
```

If your adapter takes a pre-configured client, pass a **lambda** and make sure each call provisions **isolated backing storage** (a fresh in-memory fake, a unique key prefix, or a new test database).

---

# Part B: use a store with a real agent

The next cells call the API.

## 6. Quick start with the built-in `InMemorySessionStore`

It ships with the SDK for development and tests. **Data is lost when the process exits.**

The second call passes the **same store instance** plus `resume`, so the SDK loads the transcript from the *store* instead of the local disk.

In [ ]:
from claude_agent_sdk import (
    query, ClaudeAgentOptions, InMemorySessionStore,
    AssistantMessage, ResultMessage, SystemMessage, ToolUseBlock,
)

def short(value, n=110):
    text = str(value).replace("\n", " ")
    return text if len(text) <= n else text[: n - 1] + "…"


async def ask(prompt, **opts):
    """One query in the demo folder. Returns (answer, session_id). Reports mirror errors."""
    opts.setdefault("cwd", str(DEMO_DIR))
    opts.setdefault("allowed_tools", ["Read", "Glob"])
    answer, session_id, tools = "", None, []
    try:
        async for m in query(prompt=prompt, options=ClaudeAgentOptions(**opts)):
            if isinstance(m, SystemMessage) and m.subtype == "mirror_error":
                print("   !! mirror_error:", short(m.data, 200))
            elif isinstance(m, AssistantMessage):
                tools += [b.name for b in m.content if isinstance(b, ToolUseBlock)]
            elif isinstance(m, ResultMessage):
                session_id = m.session_id
                if m.subtype == "success":
                    answer = m.result
    except Exception as error:
        print("   (query raised:", short(error, 120), ")")
    print("   tools used :", tools or "none")
    print("   answer     :", short(answer, 250))
    print("   session_id :", session_id)
    return answer, session_id


(DEMO_DIR / "notes.txt").write_text("The launch code name is PELICAN-7.\n")

mem_store = InMemorySessionStore()
_, mem_sid = await ask("Read notes.txt and tell me the code name.", session_store=mem_store)

In [ ]:
await ask(
    "What was the code name? Answer from memory. Do not use tools.",
    session_store=mem_store,
    resume=mem_sid,
)

## 7. Plug in *our* store and watch the mirror fill

Same code, different store. After the run we look straight into the folder our adapter wrote.

In [ ]:
STORE_DIR = DEMO_DIR / "store"
shutil.rmtree(STORE_DIR, ignore_errors=True)
file_store = JsonlFileSessionStore(STORE_DIR)

_, sid = await ask("Read notes.txt and tell me the code name.", session_store=file_store)

In [ ]:
from collections import Counter

print("Files our store wrote:")
for p in sorted(STORE_DIR.rglob("*.jsonl")):
    entries = [json.loads(line) for line in p.open()]
    print(" ", p.relative_to(STORE_DIR), "->", len(entries), "entries", dict(Counter(e.get("type") for e in entries)))

Notice the path: `store/<project_key>/<session_id>/__main__.jsonl`. The `project_key` is the SDK's filesystem-safe encoding of our demo folder.

### Dual-write: the local copy exists too
The subprocess **always writes locally first**, and the SDK forwards the same batch to `append()`. For a fresh session, **both copies exist**. Let's compare.

In [ ]:
import re, os

def local_transcript(session_id, cwd=DEMO_DIR):
    base = Path(os.environ.get("CLAUDE_CONFIG_DIR", Path.home() / ".claude"))
    return base / "projects" / re.sub(r"[^A-Za-z0-9]", "-", str(cwd)) / f"{session_id}.jsonl"

local = local_transcript(sid)
store_main = next(STORE_DIR.rglob("__main__.jsonl"))

print("local transcript :", local, "| exists:", local.exists())
print("local lines      :", sum(1 for _ in local.open()) if local.exists() else "n/a")
print("store entries    :", sum(1 for _ in store_main.open()))

The counts should match (or be very close). The store is a **mirror** of the local file.

## 8. Resume from the store, as if on another machine

To simulate **host B**, we point `CLAUDE_CONFIG_DIR` at an *empty* folder so no local transcript can be found. The only place the session exists is **our store**. We also pass the API key explicitly, because the fresh config directory has no saved credentials.

What the SDK does when you pass `resume` together with a store:

1. Calls `store.load(key)` **before** spawning the subprocess.
2. Writes the transcript into a **temporary config directory** and points the subprocess at it.
3. Runs the agent, then **deletes** that directory. The local copy is deleted too, so on this path **the store holds the only durable copy**.

In [ ]:
HOST_B = DEMO_DIR / "host-b-config"
shutil.rmtree(HOST_B, ignore_errors=True)
HOST_B.mkdir()

answer_b, sid_b = await ask(
    "What was the launch code name? Answer from memory only. Do not use tools.",
    session_store=file_store,
    resume=sid,
    env={"CLAUDE_CONFIG_DIR": str(HOST_B), "ANTHROPIC_API_KEY": os.environ["ANTHROPIC_API_KEY"]},
)

print("\nresumed from the store:", "pelican" in answer_b.lower())
print("host B kept a local transcript?", any(HOST_B.rglob("*.jsonl")))

If the answer contains the code name, the agent restored its history from **your store alone**. That is the multi-host story in miniature.

> **Authentication gotcha.** On a resume-from-store run the SDK seeds its temporary config directory with files from your real one. The **Python** SDK copies only credentials and `.claude.json`; the **TypeScript** SDK also copies a sanitized user `settings.json`. So in Python, an app that logs in through an `apiKeyHelper` in *user* `settings.json` fails with `Not logged in` when resuming from a store. Set `ANTHROPIC_API_KEY` in `env`, as above, or put the helper in managed or project settings.

## 9. Reading sessions back through the store

Standalone functions like `list_sessions()` read **local** files. Their store-backed twins take the store as the first argument:

| Local | Store-backed (Python) |
|---|---|
| `list_sessions` | `list_sessions_from_store(store, directory=...)` |
| `get_session_info` | `get_session_info_from_store(store, session_id)` |
| `get_session_messages` | `get_session_messages_from_store(store, session_id)` |
| `rename_session` / `tag_session` | `rename_session_via_store` / `tag_session_via_store` |
| `delete_session` | `delete_session_via_store` |
| `fork_session` | `fork_session_via_store` |
| `list_subagents` / `get_subagent_messages` | `list_subagents_from_store` / `get_subagent_messages_from_store` |

`list_sessions_from_store` works because our adapter implemented `list_sessions`.

In [ ]:
from claude_agent_sdk import list_sessions_from_store, get_session_messages_from_store

infos = await list_sessions_from_store(file_store, directory=str(DEMO_DIR))
print(f"{len(infos)} session(s) in the store")
for info in infos:
    print(" ", info.session_id[:8], "|", short(info.first_prompt, 60))

print("\nmessages in the stored session:")
for msg in await get_session_messages_from_store(file_store, sid, directory=str(DEMO_DIR)):
    print(f"  {msg.type:9}", short(msg.message, 90))

## 10. Behavior notes worth knowing

**Dual-write.** The subprocess writes locally first; the SDK forwards the batch to `append()`.
- *Fresh session, or a resume where the store had nothing*: the **local transcript outlives the run**, and the store receives a copy.
- *Run resumed from the store*: the local copy is **deleted at run end**, so the store is the only durable copy.
- To keep a fresh session from leaving a local transcript, set `CLAUDE_CONFIG_DIR` to a temp directory in `options.env`.

**Options that conflict with the mirror.** The SDK throws at startup if you combine a store with `persistSession: false` (TypeScript) or with **file checkpointing** (`enable_file_checkpointing`), whose backups go straight to local disk and are not mirrored.

**Best-effort mirroring.** Up to 3 attempts per batch, then drop and emit `mirror_error`. A call that *times out* isn't retried (the original may still land). Deduplicate by `entry.uuid`.

**`get_session_messages` returns the post-compaction chain.** After auto-compaction a store holding 503 raw entries may yield only 18 messages. For the **full raw history**, call `store.load(key)` yourself.

**`fork_session` is not a byte copy.** It reads the source entries, rewrites every `sessionId` and **remaps message UUIDs**, then appends under a new key. A backend-level copy would leave the old session id inside the transcript, so the SDK doesn't use one.

**Subagent transcripts** are mirrored under `subpath="subagents/agent-<id>"`. Implement `list_subkeys` if you want them restored on resume.

**Retention.** The SDK **never deletes** from your store. Expiry and lifecycle rules are *your* responsibility, and a run resumed from the store leaves no local transcript, so your store's retention is the only retention. Local transcripts are swept separately by `cleanupPeriodDays`.

**Optional summaries.** `list_session_summaries` lets listing read metadata in one call. Maintain it inside `append` by calling `fold_session_summary(prev, key, entries)` on each **main-transcript** batch (skip batches with a `subpath`), stamp `mtime` yourself with the same clock `list_sessions` uses, and **serialize the read-fold-write** per session. The built-in `InMemorySessionStore` is a compact reference for this.

## 11. Reference adapters

The SDK repositories ship runnable adapters (not published as packages: **copy the one closest to your backend**).

| Storage type | Model | Example |
|---|---|---|
| Object store | One part file per `append()`; `load()` lists, sorts and concatenates the parts | S3 |
| Key-value store | One list per transcript: `append()` pushes, `load()` reads a range, plus a sorted index of sessions | Redis |
| Relational / document | One row per entry as JSON, ordered by an insert-assigned key | Postgres |

Each adapter takes a **pre-configured client**, so you control credentials, TLS, region and pooling. Our `JsonlFileSessionStore` follows the same shape with a folder as the "client".

## 12. Recap

- A `SessionStore` mirrors transcripts to **your backend** so sessions survive restarts and move between hosts.
- Required: **`append`** and **`load`**. Optional: `list_sessions`, `list_session_summaries`, `delete`, `list_subkeys`.
- **Order matters; entries are opaque; `load` returns `None` for unknown sessions.**
- **Deduplicate by `uuid`** in `append`, because retries re-deliver data.
- Mirroring is **best-effort**: watch the stream for `mirror_error`.
- Validate adapters with **`run_session_store_conformance`**.
- Resume from a `cwd` matching the original run; remember the **auth gotcha** in Python.

### Try it yourself
1. Make `JsonlFileSessionStore` also implement `list_session_summaries` using `fold_session_summary`, then re-run the conformance suite (optional tests un-skip).
2. Write a **SQLite** adapter: one table `(project_key, session_id, subpath, seq INTEGER PRIMARY KEY AUTOINCREMENT, entry JSON)`.
3. Add a retention job that deletes sessions older than 7 days using `list_sessions` mtimes.
4. Run a session that uses a subagent, then inspect `list_subkeys` on your store.